# 04 · 束流参数演化 (lineplot 替代)

手册 5.5.1 菜单 1 的九图: 包络/散角/发射度/束长/能散/能量/纵向发射度 + 参考粒子轨迹。

In [ ]:
%run _bootstrap.py

In [ ]:
from astra_tools.widgets.selectors import discover_sim_runs
from astra_tools.io.astra_emit import read_emit_files, read_ref_file
runs = discover_sim_runs(SIM_DIR)
if not runs:
    raise SystemExit("工作目录没有 ASTRA 输出 — 请先运行 02_astra.ipynb")
stem = sorted(runs)[0]
print("使用 stem:", stem)
emit = read_emit_files(str(SIM_DIR / stem))
ref = read_ref_file(str(SIM_DIR / stem))
print("Xemit/Yemit/Zemit 行数:", len(emit.x.z), len(emit.y.z), len(emit.z.z))


In [ ]:
from astra_tools.plot.emit_plots import plot_lineplot_overview
plot_lineplot_overview(emit)

In [ ]:
from astra_tools.plot.emit_plots import plot_emittance_evolution, plot_energy_evolution
plot_emittance_evolution(emit)
plot_energy_evolution(emit)

In [ ]:
from astra_tools.plot.emit_plots import plot_ref_trajectory
plot_ref_trajectory(ref)

In [ ]:
# 光学函数: beta/alpha、相位推进、相干长度 (菜单 2)
from astra_tools.plot.advanced_plots import (
    plot_beta_alpha, plot_phase_advance, plot_coherence_length,
)
plot_beta_alpha(emit, ref=ref)
plot_phase_advance(emit, ref=ref)
plot_coherence_length(emit)

In [ ]:
# 核心发射度 (Cemit) 与缩减发射度 (Xemit2), 存在则显示 (菜单 4)
from astra_tools.io import parse_output_file, read_xemit2
from astra_tools.plot.advanced_plots import plot_core_emittance, plot_reduced_emittance, plot_trace_emittance
from pathlib import Path

try:
    ce = parse_output_file(str(SIM_DIR / "astra.Cemit.001"))
    plot_core_emittance(ce)
except Exception:
    print("无 Cemit 文件 (OUTPUT 中 C_EmitS=F)")
try:
    x2 = read_xemit2(str(SIM_DIR / "astra.Xemit2.001"))
    y2 = read_xemit2(str(SIM_DIR / "astra.Yemit2.001"))
    plot_reduced_emittance(x2, y2)
except Exception:
    print("无 Xemit2 文件 (OUTPUT 中 Lsub_cor=F)")

In [ ]:
# 探针轨迹与空间电荷场 (存在 track 文件时)
from astra_tools.plot.advanced_plots import plot_probe_trajectories, plot_space_charge_fields
from pathlib import Path
tr = SIM_DIR / "astra.track.001"
if tr.exists():
    plot_probe_trajectories(tr)
    plot_space_charge_fields(tr, "Ez")
else:
    print("无 track 文件 (OUTPUT 中 TrackS=F)")

In [ ]:
from astra_tools.io.astra_emit import read_sigma_file
from astra_tools.plot.emit_plots import plot_eigen_emittances
try:
    plot_eigen_emittances(read_sigma_file(str(SIM_DIR / "astra")))
except FileNotFoundError:
    print("无 Sigma 文件 (OUTPUT 中 SigmaS=F)")

## 粒子速度与平均步长 (lineplot 菜单 2)

In [ ]:
# 参考粒子速度 beta=v/c 与 gamma (由 pz 推出)
from astra_tools.plot.emit_plots import plot_velocity_evolution, plot_step_size_evolution
plot_velocity_evolution(ref)

In [ ]:
# 平均积分步长 (ref 文件相邻行的 z 间距)
plot_step_size_evolution(ref)

## 时间轴变体 (三视图 vs 时间): 把 x_axis 换成 't'

In [ ]:
from astra_tools.plot.emit_plots import (
    plot_envelope_evolution, plot_emittance_evolution,
    plot_bunch_length_evolution, plot_energy_spread_evolution)
plot_envelope_evolution(emit, x_axis="t")

In [ ]:
plot_emittance_evolution(emit, x_axis='t')

In [ ]:
plot_bunch_length_evolution(emit, x_axis='t')

### 菜单 1/2 补充: 关联能散、参考动量、压缩(时间)

In [ ]:
from astra_tools.plot.emit_plots import (
    plot_correlated_energy_spread, plot_ref_momentum)
plot_correlated_energy_spread(emit)
plot_ref_momentum(ref)

from pathlib import Path as _P
from astra_tools.io.astra_misc import read_pscan, read_scan, read_tcheck
from astra_tools.plot.advanced_plots import (
    plot_pscan_compression_time, plot_scan_fom, plot_scan_position,
    plot_tcheck_counter)
ps = SIM_DIR / (stem + ".PScan.001")
if ps.exists():
    plot_pscan_compression_time(read_pscan(ps))
else:
    print("无 PScan 文件 (NEWRUN 中 Phase_Scan=F)")
sc = SIM_DIR / (stem + ".Scan.001")
if sc.exists():
    scan = read_scan(sc)
    plot_scan_fom(scan, i=0)
    plot_scan_position(scan)
else:
    print("无 Scan 文件 (SCAN namelist 未启用)")
tc = SIM_DIR / (stem + ".tcheck.001")
if tc.exists():
    plot_tcheck_counter(read_tcheck(tc))
else:
    print("无 tcheck 文件 (TcheckS=F)")
